In [31]:
import pandas as pd
import joblib  # For loading .pkl files
import requests  # For making API calls

# Load the trained Random Forest model, scaler, and label encoder
rf_model = joblib.load('rf_model.pkl')  # Replace with the correct path if needed
scaler = joblib.load('scaler.pkl')     # Replace with the correct path if needed
label_encoder = joblib.load('label_encoder.pkl')  # Replace with the correct path if needed

# Blynk API URLs for real-time sensor values and result upload
Pulse = "https://blynk.cloud/external/api/get?token=-T_R4oiqqXLiXWihpZIJ64rUg_uDrOBK&V0"
Glucose = "https://blynk.cloud/external/api/get?token=-T_R4oiqqXLiXWihpZIJ64rUg_uDrOBK&V1"
BP_Systolic = "https://blynk.cloud/external/api/get?token=-T_R4oiqqXLiXWihpZIJ64rUg_uDrOBK&V2"
SPO2 = "https://blynk.cloud/external/api/get?token=-T_R4oiqqXLiXWihpZIJ64rUg_uDrOBK&V3"
Temperature = "https://blynk.cloud/external/api/get?token=-T_R4oiqqXLiXWihpZIJ64rUg_uDrOBK&V4"
Switch = "https://blynk.cloud/external/api/get?token=-T_R4oiqqXLiXWihpZIJ64rUg_uDrOBK&V9"

RESULT_UPLOAD_URL = "https://blynk.cloud/external/api/update?token=-T_R4oiqqXLiXWihpZIJ64rUg_uDrOBK&V7={}"

def get_sensor_value(url):
    response = requests.get(url)
    if response.status_code == 200:
        return float(response.text.strip())
    else:
        raise Exception(f"Failed to fetch data from {url}. HTTP Status Code: {response.status_code}")

def upload_result_to_blynk(result):
    response = requests.get(RESULT_UPLOAD_URL.format(result))
    if response.status_code == 200:
        print(f"Result '{result}' successfully uploaded to Blynk.")
    else:
        raise Exception(f"Failed to upload result to Blynk. HTTP Status Code: {response.status_code}")

def condition(Pulse, Glucose, BP_Systolic, SPO2, Temperature, Switch):

    # Create a DataFrame from the input values
    new_data = pd.DataFrame({
        'Pulse': [Pulse],
        'Glucose': [Glucose],
        'BP_Systolic': [BP_Systolic],
        'SPO2': [SPO2],
        'Temperature': [Temperature],
        'Switch': [Switch],

    })
    # Scale the new data using the loaded scaler
    new_data_scaled = scaler.transform(new_data)
    
    # Make predictions using the loaded Random Forest model
    dt_prediction = rf_model.predict(new_data_scaled)
    
    # Convert numerical prediction back to the original label
    result = label_encoder.inverse_transform(dt_prediction)
    
    return result[0]

# Main function to fetch real-time sensor values, make a prediction, and upload the result
if __name__ == "__main__":
    try:
        print("Fetching real-time sensor values...")
        
        # Fetch real-time sensor values
        Pulse = get_sensor_value(Pulse)
        Glucose = get_sensor_value(Glucose)
        BP_Systolic = get_sensor_value(BP_Systolic)
        SPO2 = get_sensor_value(SPO2)
        Temperature = get_sensor_value(Temperature)
        Switch = get_sensor_value(Switch)
        
        print(f"Pulse: {Pulse}")
        print(f"Glucose: {Glucose}")
        print(f"BP_Systolic: {BP_Systolic}")
        print(f"SPO2: {SPO2}")
        print(f"Temperature: {Temperature}")
        print(f"Switch: {Switch}")
        
        # Predict the disaster type
        predicted_result = condition(Pulse, Glucose, BP_Systolic, SPO2, Temperature, Switch)
        
        # Display the prediction
        print("\nPredicted Result:", predicted_result)

        # Upload the prediction result to Blynk
        upload_result_to_blynk(predicted_result)
    except Exception as e:
        print("\nError:", str(e))


Fetching real-time sensor values...
Pulse: 90.0
Glucose: 100.0
BP_Systolic: 100.0
SPO2: 95.0
Temperature: 36.0
Switch: 1.0

Predicted Result: Abnormal
Result 'Abnormal' successfully uploaded to Blynk.
